In [ ]:
# v2.0 imports — replaces systemflow.graph with systemflow.hep
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

from systemflow.hep.utils import (
    hep_graph_from_spreadsheet,
    hep_with_updated_parameters,
    dataframes_from_spreadsheet,
)
from systemflow.hep.metrics import Productivity
from systemflow.classifier import L1TClassifier, get_passed
from systemflow.metrics import precision, recall, f1_score
from systemflow.models import density_scale_model

In [ ]:
# Load spreadsheet data
run3_det, run3_proc, run3_globals = dataframes_from_spreadsheet("configurations/cms_system_60.xlsx")
run5_det, run5_proc, run5_globals = dataframes_from_spreadsheet("configurations/cms_system_200.xlsx")

In [ ]:
run5_det

In [ ]:
# Fit wall time scaling polynomial for CPU and GPU runtimes
scaling = pd.read_excel("wall time scaling.xlsx", sheet_name="Data")
fit_poly = lambda x, k3, k2, k1: k3 * x ** 3 + k2 * x ** 2 + k1 * x
k, cv = curve_fit(fit_poly, scaling["Size"], scaling["Wall Time"])
k_gpu, cv_gpu = curve_fit(fit_poly, scaling["Size"], scaling["Wall Time GPU"])

In [ ]:
# Define complexity functions for CPU and GPU trigger runtimes
funcs = {"Global": lambda x: fit_poly(x, *k), "Intermediate": lambda x: x / 2.0e6}
funcs_gpu = {"Global": lambda x: fit_poly(x, *k_gpu), "Intermediate": lambda x: x / 2.0e6}

In [ ]:
# Build base graph definitions
baseline_r3_def = hep_graph_from_spreadsheet("configurations/cms_system_60.xlsx", functions=funcs)
baseline_def = hep_graph_from_spreadsheet("configurations/cms_system_200.xlsx", functions=funcs)
gpu_def = hep_graph_from_spreadsheet("configurations/cms_system_200.xlsx", functions=funcs_gpu)

In [ ]:
# Build variant graph definitions using parameter updates
# (replaces deepcopy + manual attribute modification + update_throughput)

# baseline_2: Run-5 with L1T reduction ratio changed to 400
baseline_2_def = hep_with_updated_parameters(baseline_def, {
    "Intermediate": {"reduction ratio (1)": 400}
})

# L1T tracking skill boost (CPU)
l1t_classifier = L1TClassifier(skill_boost=0.40)
l1t_def = hep_with_updated_parameters(baseline_def, {
    "Intermediate": {"classifier (obj)": l1t_classifier}
})

# Smart pixels: Inner Tracker data reduced by 54%
it_data = baseline_def.get_node("Inner Tracker").parameters["sample data (B)"]
smpx_def = hep_with_updated_parameters(baseline_def, {
    "Inner Tracker": {"sample data (B)": it_data * (1 - 0.54)}
})

# GPU + L1T tracking
l1t_classifier_gpu = L1TClassifier(skill_boost=0.40)
gpu_l1t_def = hep_with_updated_parameters(gpu_def, {
    "Intermediate": {"classifier (obj)": l1t_classifier_gpu}
})

# Smart pixels + L1T tracking (CPU)
l1t_classifier_smpx = L1TClassifier(skill_boost=0.40)
smpx_l1t_def = hep_with_updated_parameters(baseline_def, {
    "Inner Tracker": {"sample data (B)": it_data * (1 - 0.54)},
    "Intermediate": {"classifier (obj)": l1t_classifier_smpx}
})

# GPU + Smart pixels
it_data_gpu = gpu_def.get_node("Inner Tracker").parameters["sample data (B)"]
gpu_smpx_def = hep_with_updated_parameters(gpu_def, {
    "Inner Tracker": {"sample data (B)": it_data_gpu * (1 - 0.54)}
})

# GPU + Smart pixels + L1T tracking
l1t_classifier_gpu_smpx = L1TClassifier(skill_boost=0.40)
gpu_smpx_l1t_def = hep_with_updated_parameters(gpu_def, {
    "Inner Tracker": {"sample data (B)": it_data_gpu * (1 - 0.54)},
    "Intermediate": {"classifier (obj)": l1t_classifier_gpu_smpx}
})

In [ ]:
# Execute all graph definitions to produce results
baseline_r3 = baseline_r3_def()
baseline = baseline_def()
baseline_2 = baseline_2_def()
gpu = gpu_def()
l1t = l1t_def()
smpx = smpx_def()
gpu_l1t = gpu_l1t_def()
smpx_l1t = smpx_l1t_def()
gpu_smpx = gpu_smpx_def()
gpu_smpx_l1t = gpu_smpx_l1t_def()

In [ ]:
# Inspect baseline results
baseline.metric_values

In [ ]:
# Inspect a specific node
baseline.get_node("Intermediate").properties

In [ ]:
def extract_results(graph):
    """Extract power, precision, recall, F1, and productivity from an executed v2.0 graph."""
    mv = graph.metric_values
    power = mv["total power (W)"] / density_scale_model(2032)
    p = mv["precision (%)"]
    r = mv["recall (%)"]
    f1 = mv["f1 score (%)"]
    contingency = mv["pipeline contingency (2x2)"]
    prod = f1 * np.sum(get_passed(contingency)) / power
    return power, p, r, f1, prod

In [ ]:
# Quick comparison: baseline_r3 vs baseline_2 vs baseline
conditions = [baseline_r3, baseline_2, baseline]
pileup = np.array([60, 200, 200])[:, np.newaxis]
rejection = np.array([400, 400, 53])[:, np.newaxis]

cond_results = np.stack([extract_results(g) for g in conditions])
cond_results = np.concatenate((pileup, rejection, cond_results), axis=1)

df2 = pd.DataFrame(cond_results, columns=[
    "Pileup", "L1T Reduction Ratio", "Power (W)",
    "Accuracy (%)", "Recall (%)", "F1 Score (%)",
    "Productivity (Relevant Samples/J)"
])
df2

In [ ]:
# Full results table across all 9 configurations
all_graphs = [baseline_r3, baseline, gpu, l1t, smpx, gpu_l1t, smpx_l1t, gpu_smpx, gpu_smpx_l1t]

pileup = np.array([[60, 200, 200, 200, 200, 200, 200, 200, 200]])
rejection = np.array([[400, 53, 53, 53, 53, 53, 53, 53, 53]])
has_gpu = [False, False, True, False, False, True, False, True, True]
has_smpx = [False, False, False, False, True, False, True, True, True]
has_l1t = [False, False, False, True, False, True, True, False, True]

results = np.stack([extract_results(g) for g in all_graphs])
results = np.concatenate((pileup, rejection, np.transpose(results)), axis=0)

df = pd.DataFrame(results.transpose(), columns=[
    "Pileup", "L1T Reduction Ratio", "Power (W)",
    "Accuracy (%)", "Recall (%)", "F1 Score (%)",
    "Productivity (Relevant Samples/J)"
])
df["GPU HLT"] = has_gpu
df["L1T Tracking"] = has_l1t
df["Smart Sensors"] = has_smpx

In [ ]:
df

In [ ]:
df["Productivity (Relevant Samples/J)"] * 1e3

In [ ]:
df.to_excel("experimental_table.xlsx", index=False)